# 酒店评论智能问答系统

本项目直接使用前面在 `酒店评论知识库构建.ipynb` 中构建好的多个知识库（DashVector评论向量库、反向Query向量库、ChromaDB摘要知识库、以及BM25倒排索引），实现一个完整的 RAG（检索增强生成）问答系统。

## 1. 导入库与环境配置

In [1]:
import os
import re
import math
import json
import pickle
import jieba
import nltk
from collections import Counter
from pathlib import Path
from tqdm.notebook import tqdm

# 相关 API 客户端
import dashscope
from dashscope import TextEmbedding, Generation
import dashvector
import chromadb

# 设置项目路径
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
print(f"离线数据目录: {DATA_DIR}")

# 检查环境变量
required_env = {
    "DASHSCOPE_API_KEY": os.getenv("DASHSCOPE_API_KEY"),
    "DASHVECTOR_API_KEY": os.getenv("DASHVECTOR_API_KEY"),
    "DASHVECTOR_HOTEL_ENDPOINT": os.getenv("DASHVECTOR_HOTEL_ENDPOINT"),
}

missing = [k for k, v in required_env.items() if not v]
if missing:
    print(f"警告：缺少环境变量: {', '.join(missing)}。请确保已经配置好 API Key。")
else:
    print("环境变量检测成功:")
    for key, value in required_env.items():
        print(f"- {key}: {value[:10]}...{value[-4:]}")
        
# 全局设置 DashScope API Key
dashscope.api_key = required_env.get("DASHSCOPE_API_KEY")

离线数据目录: c:\Users\syh\jupyter\llm course\Exp3\data
环境变量检测成功:
- DASHSCOPE_API_KEY: sk-1ac4819...4ca3
- DASHVECTOR_API_KEY: sk-i9DfXVC...8A0A
- DASHVECTOR_HOTEL_ENDPOINT: vrs-cn-adx....com


## 2. 工具类及向量客户端初始化定义
包含在原过程中使用的 `EmbeddingClient` 以及 `InvertedIndex`。我们需要用这些工具加载保存过的索引和数据库。

In [ ]:
# EmbeddingClient 和 InvertedIndex 已在 lib_syh 中定义，此处无需重复


## 3. RAG 核心系统定义

In [ ]:
from lib_syh import HotelReviewRAG, print_rag_result


## 4. 实例化系统并进行测试
模拟 `演示示例.ipynb` 过程，我们将运行几个 QA 样例。

In [6]:
# 初始化 RAG 系统
rag_system = HotelReviewRAG(
    api_key=required_env.get("DASHSCOPE_API_KEY", ""),
    dashvector_api_key=required_env.get("DASHVECTOR_API_KEY", ""),
    dashvector_endpoint=required_env.get("DASHVECTOR_HOTEL_ENDPOINT", ""),
    data_dir=DATA_DIR
)

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\syh\AppData\Local\Temp\jieba.cache


DashVector 数据库 (评论/反向Query) 链接成功
ChromaDB 摘要数据库链接成功


Loading model cost 1.130 seconds.
Prefix dict has been built successfully.


倒排索引已成功加载，包含 2171 篇文档和 10734 个词项


In [7]:
# 示例 1
test_query = "酒店的床舒服吗？睡眠质量怎么样？"
result1 = rag_system.query(test_query, enable_hyde=False)
print_rag_result(result1)

正在检索信息...
正在生成回答...
💡 客户提问：酒店的床舒服吗？睡眠质量怎么样？
------------------------------------------------------------
📚 召回的知识库上下文：
  [1] 床好，水好，服务好。早餐丰盛，而且国际化，又有本地特色。十分开心。
  [2] 花园酒店，开业38年，经久不衰，大堂永远人庭若市，伴随着悠扬的钢琴声，那么的和谐高雅。此次专门接远道而归的家人，酒店的服务没有过分的热情，确也非常的周到和细致见内涵。房间由原定的红棉套房升级了到了新装...
  [3] 根据403条评论，该酒店的卫生状况呈现出显著的两极分化。大量负面评论集中反映了严重的卫生问题，尤其在老旧客房区域。最突出的问题是细节清洁的严重缺失：多位住客报告在床单、枕套、地毯上发现头发、碎屑甚至不...
  [4] 交通方便，去哪很方便，服务很好，房间设施虽有点陈旧但设计很合理，床超级超级舒服大赞！瀑布餐厅人很多感受不好，但二楼荔湾粤式餐厅推荐，环境菜品都很好～下次还来
  [5] 根据105条关于退房/入住效率的评论，客人的体验呈现明显的两极分化。主要问题集中在入住环节：大量客人反映排队等待时间过长，普遍需要15-30分钟，高峰期甚至超过一小时，且大堂缺乏足够的等候座位。前台效...
  [6] 在性价比类别下，156条评论呈现出显著的分化与核心共识。多数评论者认为该酒店在五星级酒店中价格相对亲民，尤其是在非节假日或通过携程等平台购买包含早餐、行政礼遇及餐饮抵用券的套餐时，普遍感到“物超所值”...
  [7] 一如既往的好，喜欢。床特别舒服。环境也好。早餐也丰富
  [8] 住了十几天，感觉很舒服。空调干湿度控制的很好，不会像一些酒店太干燥。床褥松软温暖，很适合睡眠。早餐很棒。
  [9] 总体还是很满意的，酒店后花园很美，大堂气派有老牌五星的风格，酒店门口一直有出租车等候外出很方便。房间洗澡水大，床品很舒服睡眠很好，提供质量很好的肥皂我很喜欢，现在很多酒店都是洗手液。饮用水也很不错，服...
------------------------------------------------------------
🤖 智能助手回答：

是的，酒店的床普遍受到住客高度好评，睡眠体验整体非常出色。根据多位住客反馈：

✅ *

In [8]:
# 示例 2
test_query2 = "听说这家酒店有个瀑布花园，值得去看看吗？带孩子去玩如何？"
result2 = rag_system.query(test_query2, enable_hyde=False)
print_rag_result(result2)

正在检索信息...
正在生成回答...
💡 客户提问：听说这家酒店有个瀑布花园，值得去看看吗？带孩子去玩如何？
------------------------------------------------------------
📚 召回的知识库上下文：
  [1] 名副其实的花园酒店，瀑布餐厅，庭院都很有特色哦

卫生：干净
服务：很好

设施：很好
  [2] 本次带小孩入住，非常满意   整个酒店的环境  很惊艳，尤其是瀑布花园和餐厅，非常棒。
  [3] 因为团了两份一晚的套餐，忘记复制了，竟然要写两遍哈哈??我不想写两遍～这篇就看图说话吧
图1花园瀑布，好看的值得一逛里面还有很多小景点出片的，就有种影楼拍照的密集布景～很嗲
图2每天晚上回去会有小甜品...
  [4] 酒店特别好，里面有个花园，可以带宝贝去喂喂鱼，逛逛花园，去北京路和码头都不远，我们都是打车前往
  [5] 关于酒店的公共设施，评论普遍给予极高评价，认为其是酒店最核心的亮点与竞争力。主要观点集中在以下几个方面：

首先，酒店内独具特色的岭南园林花园是几乎所有评论都提及的焦点。住客们惊叹于在市中心竟能拥有如...
  [6] 根据403条评论，该酒店的卫生状况呈现出显著的两极分化。大量负面评论集中反映了严重的卫生问题，尤其在老旧客房区域。最突出的问题是细节清洁的严重缺失：多位住客报告在床单、枕套、地毯上发现头发、碎屑甚至不...
  [7] 广州老字号酒店，服务很好，酒店很漂亮很大气，办理入住的小哥哥很亲切看我们带孩子还帮我升级了房间，下次来广州还会来住这家
  [8] 酒店过年气氛很好，酒店环境不错，有瀑布花园，适合全家出游
  [9] 在景观/朝向方面，该酒店的突出优势在于其独特且丰富的内外景观。内部景观以标志性的瀑布花园为核心，多数评论称赞房间窗户正对瀑布与花木，营造出“花团锦簇”、“春意盎然”的园林意境，后花园的岭南园林设计、小...
------------------------------------------------------------
🤖 智能助手回答：

是的，这家酒店的瀑布花园非常值得一看，尤其适合带孩子一同游玩！😊

✅ **亮点突出**：  
- 花园是酒店最具标志性的景观，被多位住客誉为“城市绿洲”“世外桃源”，核心包括**人工双瀑布、小桥流水、锦鲤